# LH Nautical – Data Analysis

Autor: Luciano Brandao 

Este notebook apresenta a análise da LH Nautical com o objetivo de
organizar dados, gerar insights de negócio e propor modelos preditivos. Os datasets disponibilizados pela companhia, incluem dados sobre vendas, produtos, clientes e custos. 

## 1. Exploração Inicial (EDA)
Objetivo: entender a estrutura, distribuição e qualidade dos dados de vendas antes de qualquer tratamento.

In [22]:
import pandas as pd

df = pd.read_csv("../01_data/raw/vendas_2023_2024.csv")
df.head()

,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


### Parte 1 — Visão geral do dataset Vendas
Objetivo: entender o volume e o período dos dados disponíveis.
- Quantidade total de linhas: 9895  
- Quantidade total de colunas: 6  
- Intervalo de datas: de 01/01/2023 até 31/12/2024  

In [4]:
num_linhas = df.shape[0]
num_colunas = df.shape[1]

df['sale_date'] = pd.to_datetime(df['sale_date'], errors='coerce')

data_min = df['sale_date'].min()
data_max = df['sale_date'].max()

num_linhas, num_colunas, data_min, data_max

(9895, 6, Timestamp('2023-01-01 00:00:00'), Timestamp('2024-12-31 00:00:00'))

### Parte 2 — Análise da coluna total

Objetivo: entender a distribuição básica dos valores de vendas.

- Valor mínimo: 294.5  
- Valor máximo: 2222973.0  
- Valor médio: 263797.8  

In [5]:
min_total = df['total'].min()
max_total = df['total'].max()
mean_total = df['total'].mean()

min_total, max_total, mean_total

(np.float64(294.5), np.float64(2222973.0), np.float64(263797.82826680137))

### Parte 3 — Interpretação

O dataset apresenta boa estrutura inicial, com volume relevante de dados e período consistente de análise.

Observa-se grande variação nos valores de venda, com presença de possíveis outliers, o que pode impactar análises futuras. A coluna de datas apresenta formatos inconsistentes, o que pode gerar falhas na interpretação e análise temporal dos dados. De forma geral, o dataset não está pronto para uso direto e exige tratamento prévio antes de análises mais avançadas.

## 2. Limpeza e Tratamento de Dados



#### Carregamento dataset Produtos  

In [7]:
df_prod = pd.read_csv("../01_data/raw/produtos_raw.csv")

df_prod.head()

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


#### Parte 1 — Padronização de categorias produtos em: eletrônicos, propulsão e ancoragem

In [8]:
df_prod['actual_category'].value_counts()

actual_category
AncorageM                9
Propução                 8
Ancoraguem               8
Eletronicoz              7
eletrônicos              7
ELETRONICOS              6
E L E T R Ô N I C O S    6
PROPULSAO                6
P R O P U L S Ã O        6
propulsão                6
Eletrunicos              5
eLeTrÔnIcOs              5
Propulção                5
Prop                     5
Propulssão               5
Encoragem                5
Ancorajm                 5
A N C O R A G E M        5
aNcOrAgEm                5
Eletrônicos              4
propulsao                4
eletronicos              3
EletrônicoS              3
Propulçao                3
Ancoragem                3
Ancorajem                3
Eletroniscos             2
Eletronicos              2
pRoPuLsÃo                2
Propulsam                2
AnCoRaGeM                2
ancoragem                2
Ancorajen                2
ELEtRÔNICOS              1
PrOpUlSãO                1
ANCORAGEM                1
Encoragi    

In [41]:
def padronizar_categoria(cat):
    cat = str(cat).lower().replace(" ", "").strip()
    
    if 'eletr' in cat:
        return 'eletrônicos'
    elif 'prop' in cat:   
        return 'propulsão'
    elif 'ancor' in cat or 'encor' in cat:
        return 'ancoragem'
    else:
        return cat

df_prod['actual_category'] = df_prod['actual_category'].apply(padronizar_categoria)

In [17]:
df_prod['actual_category'].value_counts()

actual_category
propulsão      53
ancoragem      53
eletrônicos    51
Name: count, dtype: int64

#### Parte 2 — Conversão valores para tipo numérico

In [18]:
df_prod['price'] = (
    df_prod['price']
    .str.replace('R$', '', regex=False)
    .str.strip()
    .astype(float)
)

In [19]:
df_prod.dtypes

name                   str
price              float64
code                 int64
actual_category        str
dtype: object

#### Parte 3 — Remoção de duplicatas

In [42]:
# Duplicatas
num_antes = df_prod.shape[0]

df_prod = df_prod.drop_duplicates()

num_depois = df_prod.shape[0]

duplicados_removidos = num_antes - num_depois

print(duplicados_removidos)

7


In [21]:
df_prod.to_csv("../01_data/processed/produtos_tratados.csv", index=False)

### Parte 4 - Normalização arquivo formato JSON p CSV  

#### Carregamento Dados Custo Importação    

In [48]:
df_custos = pd.read_json("../01_data/raw/custos_importacao.json")

df_custos.head()

,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105..."
1,2,Transponder Furuno Marlin,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432..."
2,3,Radar Furuno Pulse Leviathan,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254..."
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909..."
4,5,Piloto Automático Furuno Storm,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600..."


In [49]:
import pandas as pd

# Carregar JSON
df = pd.read_json("../01_data/raw/custos_importacao.json")

# Explodir lista
df = df.explode("historic_data")

# Expandir JSON aninhado
df_hist = pd.json_normalize(df["historic_data"])

# Juntar tudo
df_final = pd.concat(
    [
        df[["product_id", "product_name", "category"]].reset_index(drop=True),
        df_hist.reset_index(drop=True)
    ],
    axis=1
)

# Salvar CSV
df_final.to_csv("../01_data/processed/custos_importacao_tratado.csv", index=False)

# Validação
print(len(df_final))

1260


## 3. Análise de Vendas

###  Análise de Resultado Lucro/Prejuízo por Produto no período

#### Parte 1 — Agregação Dados e Cálculos

##### •	Calculo custo total em BRL por transação
##### •	Identificação transações com prejuízo
##### •	Agregação dados por id_produto, gerando:
Receita total (BRL), Prejuízo total (BRL) e % de perda (prejuízo_total / receita_total)





####
#### Parte 2 — Análise visual

## 4. Análise de Clientes

## 5. Previsão de Demanda

## 6. Sistema de Recomendação

## 7. Insights de Negócio

## 8. Conclusão